In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import mlflow
import dagshub
from dotenv import load_dotenv
import joblib, os

In [2]:
load_dotenv()
dagshub.init(repo_owner='IbrahimFaye', repo_name='weather-agri', mlflow=True)
mlflow.set_experiment("besoin_irrigation")

DATA_PATH = "../data/irrigation_dataset.csv"
df = pd.read_csv(DATA_PATH)

Accessing as IbrahimFaye

Initialized MLflow to track repo "IbrahimFaye/weather-agri"

Repository IbrahimFaye/weather-agri initialized!

In [7]:
df.head()

,lat,lon,observation_time,temp_c,humidity,precipitation_mm,wind_speed,pressure,hour,day,month,temp_rolling_mean,humidity_rolling_mean,precip_rolling_sum,wind_rolling_mean,et0,cum_rain_3days,water_need,irrigation_label
0,14.727592,-16.911621,2022-01-01 00:00:00+00:00,24.0,34.0,0.0,12.9,1013.1,0,1,1,24.000000,34.0,0.0,12.900000,0.139016,0.0,0.318347,low
1,14.727592,-16.911621,2022-01-01 01:00:00+00:00,22.7,32.0,0.0,12.0,1013.0,1,1,1,23.350000,33.0,0.0,12.450000,0.139016,0.0,0.305835,low
2,14.727592,-16.911621,2022-01-01 02:00:00+00:00,22.4,24.0,0.0,11.8,1012.9,2,1,1,23.033333,30.0,0.0,12.233333,0.116954,0.0,0.254959,low
3,14.727592,-16.911621,2022-01-01 03:00:00+00:00,22.7,19.0,0.0,13.0,1012.4,3,1,1,22.600000,25.0,0.0,12.266667,0.051020,0.0,0.117347,low
4,14.727592,-16.911621,2022-01-01 04:00:00+00:00,22.7,17.0,0.0,14.8,1011.9,4,1,1,22.600000,20.0,0.0,13.200000,0.051020,0.0,0.126530,low


In [13]:
print(df.describe())

                lat           lon        temp_c      humidity  \
count  78912.000000  78912.000000  78912.000000  78912.000000   
mean      14.985354    -16.127482     27.555796     51.388940   
std        1.134025      0.555957      5.120990     27.878398   
min       13.743409    -16.911621     13.700000      3.000000   
25%       13.743409    -16.911621     24.100000     25.000000   
50%       14.727592    -15.785126     26.700000     50.000000   
75%       16.485060    -15.685699     31.000000     77.000000   
max       16.485060    -15.685699     46.200000    100.000000   

       precipitation_mm    wind_speed      pressure         hour  \
count      78912.000000  78912.000000  78912.000000  78912.00000   
mean           0.060325     12.292882   1011.623372     11.50000   
std            0.519937      5.199194      2.112629      6.92223   
min            0.000000      0.000000   1002.300000      0.00000   
25%            0.000000      8.500000   1010.300000      5.75000   
50%   

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78912 entries, 0 to 78911
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lat                    78912 non-null  float64
 1   lon                    78912 non-null  float64
 2   observation_time       78912 non-null  object 
 3   temp_c                 78912 non-null  float64
 4   humidity               78912 non-null  float64
 5   precipitation_mm       78912 non-null  float64
 6   wind_speed             78912 non-null  float64
 7   pressure               78912 non-null  float64
 8   hour                   78912 non-null  int64  
 9   day                    78912 non-null  int64  
 10  month                  78912 non-null  int64  
 11  temp_rolling_mean      78912 non-null  float64
 12  humidity_rolling_mean  78912 non-null  float64
 13  precip_rolling_sum     78912 non-null  float64
 14  wind_rolling_mean      78912 non-null  float64
 15  et

In [7]:
####Calcul d'une estimation simple d'évapotranspiration (ET0)
df["et0"] = 0.0023 * np.sqrt(np.maximum(df["temp_c"].rolling(3).max() - df["temp_c"].rolling(3).min(), 0)) * (df["temp_c"] + 17.8)
df["et0"].fillna(df["et0"].mean(), inplace=True)

#####Calcul du besoin brut en eau (mm/jour)
df["water_need"] = (df["et0"] * (1 + (df["wind_speed"] / 10))) - df["precip_rolling_sum"]
df["water_need"] = df["water_need"].clip(lower=0)

drop_cols = ["observation_time", "irrigation_label"]

X = df.drop(columns=drop_cols + ["water_need"])
y = df["water_need"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def train_irrigation_model():
    with mlflow.start_run(run_name="RandomForestRegressor_ib") as run:

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        print(f"💧 Irrigation Model — MAE: {mae:.3f} | R²: {r2:.3f}")

        mlflow.log_metrics({"mae": mae, "r2": r2})
        
        os.makedirs("artifacts", exist_ok=True)
        joblib.dump(model, "artifacts/irrigation_model.pkl", compress=3)
        mlflow.log_artifact("artifacts/irrigation_model.pkl", artifact_path="model")

        importances = dict(zip(X.columns, model.feature_importances_))
        mlflow.log_dict(importances, "feature_importance.json")

if __name__ == "__main__":
    train_irrigation_model()


C:\Users\USER\AppData\Local\Temp\ipykernel_16292\1051100159.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["et0"].fillna(df["et0"].mean(), inplace=True)


💧 Irrigation Model — MAE: 0.002 | R²: 0.999
🏃 View run RandomForestRegressor_ib at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/5/runs/b4fb021bf8674f8f911ae3ecc903e526
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/5
